# CIDS · 05 — S2b: SAM-Med3D-turbo trained on realistic + negative boxes

S2a trained SAM only on near-perfect boxes (GT ± 5 voxels). In the automatic pipeline SAM gets **SegResNet's candidate boxes**: too big, shifted, partly covering the tumour, or around tissue that is **not tumour at all**. S2b trains the same small decoder (LoRA r=8 + 3D box tokens, encoder frozen, cached embeddings) on three box types:

| box | share of box prompts | target |
|---|---|---|
| clean (GT ± 5 vox) | 40% | tumour |
| messy (0.7–1.8× size, shifted ≤ 40%) | 40% | tumour |
| negative (on tumour-free tissue) | 20% | **empty** → SAM learns "no tumour here" |

Best epoch = mean(clean-box Dice, messy-box Dice, negative rejection rate) on val. Test also evaluates the **S2a checkpoint** on the same boxes (`compare`).

`RUN = 's2b'` is the new model. `RUN = 's2a100'` re-trains S2a for 100 epochs with clean boxes only — the fair baseline (same budget), can run on the second Kaggle account.

**Settings:** GPU T4 x2 · **Internet ON** · Persistence: Files.  
**Inputs:** `cids-sam-emb` (or the S0 output), the 2-channel cache, and `cids-checkpoints` (for `sam_lora_best.pt` = S2a).  
**Run:** smoke test first, then *Save Version → Save & Run All* (~1–1.5 h).

In [ ]:
RUN = 's2b'                 # 's2b' (realistic + negative boxes)  or  's2a100' (clean boxes, fair baseline)
TUNE = 'lora'

REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

EMB_ROOT    = ''              # '' = auto-detect cids-sam-emb
CACHE_ROOT  = ''              # '' = auto-detect the 2-channel cache
S2A_CKPT    = ''              # '' = auto-detect sam_lora_best.pt (cids-checkpoints) -> tested as 'compare'
RESUME_FROM = ''
EPOCHS = 100

BOX_AUG = 'realistic' if RUN == 's2b' else 'clean'
CODE_DIR = '/kaggle/working/cids'
OUT      = f'/kaggle/working/{RUN}_{TUNE}'


In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not).
    Progress bars redraw a line with '\r'; only each line's final state is printed (one line per epoch)."""
    import subprocess
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf = b''
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        buf += chunk
        *lines, buf = buf.split(b'\n')
        for line in lines:
            print(line.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if buf.strip():
        print(buf.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')


In [ ]:
sh(f"""pip install -q medim einops appdirs""")
sh(f"""pytest -q tests""")

In [ ]:
import sys, torch, pandas as pd
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
p = torch.cuda.get_device_properties(0); print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')
sys.path.insert(0, CODE_DIR)
sys.path.insert(0, f'{CODE_DIR}/scripts')
from s2a_sam_finetune import find_emb_root
from cids.data.cache import find_cache_root
def _counts(r):
    return len(list(Path(r, 'train').glob('*.npz'))), len(list(Path(r, 'test').glob('*.npz')))
try:
    EMB_ROOT = str(find_emb_root(EMB_ROOT))
except FileNotFoundError:
    EMB_ROOT = ''
if not EMB_ROOT or _counts(EMB_ROOT) != (1200, 306):   # no folder, or the incomplete loose-file copy
    import zipfile                                       # -> use the complete sam_emb.zip (~3 min to unzip)
    z = next(Path('/kaggle/input').rglob('sam_emb.zip'))
    dst = Path('/kaggle/tmp/sam_emb_unzipped')
    if not dst.exists():
        print('unzipping', z, '->', dst); zipfile.ZipFile(z).extractall(dst)
    EMB_ROOT = str(find_emb_root('', str(dst)))
n_tr, n_te = _counts(EMB_ROOT)
print('embeddings:', EMB_ROOT, '|', n_tr, 'train /', n_te, 'test'); assert (n_tr, n_te) == (1200, 306)
try:
    CACHE_ROOT = str(find_cache_root(CACHE_ROOT)); print('cache:', CACHE_ROOT)
except FileNotFoundError:
    CACHE_ROOT = 'none'; print('!! 2-channel cache not attached -> only crop-level Dice will be reported')
sp = pd.read_csv('configs/splits.csv'); print(sp.split.value_counts().to_dict())
if not S2A_CKPT:
    hits = list(Path('/kaggle/input').rglob('sam_lora_best.pt'))
    S2A_CKPT = str(hits[0]) if hits else ''
print('S2a checkpoint (compare):', S2A_CKPT or '-- not attached, comparison skipped')
COMPARE = f'--compare_ckpt {S2A_CKPT}' if S2A_CKPT else ''
print('run:', RUN, '| box_aug:', BOX_AUG, '| out:', OUT)

## Smoke test — 16 train, 1 epoch, 8 val + 8 test (~3 min)
Check: no errors; the val line now also prints `messy` and `neg_reject`; the test prints a **Negative boxes** block for `finetuned` and `compare`. After 1 short epoch the numbers are not meaningful yet.

In [ ]:
sh(f"""python scripts/s2a_sam_finetune.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --tune {TUNE} --box_aug {BOX_AUG} {COMPARE} --out /kaggle/working/s2b_smoke --epochs 1 --val_every 1 --limit_train 16 --limit_eval 8""")
sh(f"""rm -rf /kaggle/working/s2b_smoke""")


## Full run (~1–1.5 h) — use *Save Version → Save & Run All*

In [ ]:
resume = f'--resume_from {RESUME_FROM}' if RESUME_FROM else ''
sh(f"""python scripts/s2a_sam_finetune.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --tune {TUNE} --box_aug {BOX_AUG} {COMPARE} --out {OUT} --epochs {EPOCHS} {resume}""")


In [ ]:
import json, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUT}/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(log.epoch, log.loss); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
v = log.dropna(subset=['val_score'])
for c in ['val_box1', 'val_messy1', 'val_neg_reject', 'val_click1']:
    if c in v: ax[1].plot(v.epoch, v[c], marker='o', label=c[4:])
ax[1].legend(); ax[1].set_title('val (120, crop)'); plt.tight_layout(); plt.show()
print(json.dumps(json.load(open(f'{OUT}/test_summary.json')), indent=2))
sh(f"""du -sh {OUT}""")
